### Fetching LeetCode Submission Data

This section will fetch your LeetCode submission data using an unofficial GraphQL endpoint. You'll be prompted to enter your LeetCode username.

In [1]:
import requests
import pandas as pd
from IPython.display import display

YOUR_LEETCODE_USERNAME = input("Please enter your LeetCode username: ")

LEETCODE_API_URL = "https://leetcode.com/graphql/"

query = """
query getUserStats($username: String!) {
  matchedUser(username: $username) {
    submitStats: submitStatsGlobal {
      acSubmissionNum {
        difficulty
        count
        submissions
      }
      totalSubmissionNum {
        difficulty
        count
        submissions
      }
    }
  }
}
"""

variables = {
    "username": YOUR_LEETCODE_USERNAME
}

print(f"Fetching stats for user: {YOUR_LEETCODE_USERNAME}...")

response = requests.post(LEETCODE_API_URL, json={'query': query, 'variables': variables})
response_data = response.json()

if 'errors' in response_data:
    print("Error fetching data:", response_data['errors'])
elif response_data.get('data', {}).get('matchedUser') is None:
    print(f"Could not find user '{YOUR_LEETCODE_USERNAME}'. Please check the username.")
else:
    stats_data = response_data['data']['matchedUser']['submitStats']

    ac_subs = stats_data['acSubmissionNum']
    total_subs = stats_data['totalSubmissionNum']

    combined_stats = []
    for ac, total in zip(ac_subs, total_subs):
        combined_stats.append({
            "Difficulty": ac['difficulty'],
            "Problems Solved (AC)": ac['count'],
            "Total Submissions (All)": total['submissions']
        })

    df_stats = pd.DataFrame(combined_stats)

    print("\nUser Stats DataFrame:")
    display(df_stats)

Please enter your LeetCode username: pradneshputpat
Fetching stats for user: pradneshputpat...

User Stats DataFrame:


,Difficulty,Problems Solved (AC),Total Submissions (All)
0,All,345,969
1,Easy,159,365
2,Medium,166,532
3,Hard,20,72


In [2]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from IPython.display import display

LEETCODE_SESSION = ""
CSRF_TOKEN = ""

headers = {
    'Cookie': f'LEETCODE_SESSION={LEETCODE_SESSION}; csrftoken={CSRF_TOKEN};',
    'X-csrftoken': CSRF_TOKEN,
    'Content-Type': 'application/json',
    'Referer': 'https://leetcode.com/'
}

LEETCODE_API_URL = "https://leetcode.com/graphql/"

query = """
query submissionList($offset: Int!, $limit: Int!) {
  submissionList(offset: $offset, limit: $limit) {
    hasNext
    submissions {
      id
      title
      titleSlug
      statusDisplay
      timestamp
    }
  }
}
"""

all_submissions = []
offset = 0
limit = 20
has_next = True

print("Fetching full submission history. This might take a minute depending on your volume...")

with tqdm(unit=" pages") as pbar:
    while has_next:
        variables = {
            "offset": offset,
            "limit": limit
        }

        response = requests.post(LEETCODE_API_URL, json={'query': query, 'variables': variables}, headers=headers)

        if response.status_code != 200:
            print(f"Failed to fetch data. Check your cookies. Status Code: {response.status_code}")
            break

        response_data = response.json()

        if 'errors' in response_data:
            print("Error:", response_data['errors'])
            break

        data = response_data.get('data', {}).get('submissionList')
        if not data:
            print("Could not retrieve submission list. Ensure your LEETCODE_SESSION is valid and not expired.")
            break

        submissions = data.get('submissions', [])
        all_submissions.extend(submissions)

        has_next = data.get('hasNext', False)
        offset += limit
        pbar.update(1)

        time.sleep(0.5)

print(f"\nFetched {len(all_submissions)} total submissions.")

if all_submissions:
    df = pd.DataFrame(all_submissions)

    df['date'] = pd.to_datetime(df['timestamp'], unit='s')

    df_ac = df[df['statusDisplay'] == 'Accepted'].copy()

    if not df_ac.empty:
        first_solves = df_ac.groupby(['titleSlug', 'title'])['date'].min().reset_index()

        first_solves.rename(columns={'date': 'first_solved_date'}, inplace=True)

        distinct_2026_problems = first_solves[first_solves['first_solved_date'].dt.year == 2026].copy()

        distinct_2026_problems.sort_values(by='first_solved_date', inplace=True)

        print("\n" + "-"*50)
        print(f"Total DISTINCT new problems solved strictly in 2026: {len(distinct_2026_problems)}")
        print("-" * 50)

        print("\nHere are the specific problems you conquered this year:")
        display(distinct_2026_problems.reset_index(drop=True))
    else:
        print("No accepted submissions found in your history.")

Fetching full submission history. This might take a minute depending on your volume...


49 pages [00:38,  1.26 pages/s]


Fetched 969 total submissions.

--------------------------------------------------
Total DISTINCT new problems solved strictly in 2026: 91
--------------------------------------------------

Here are the specific problems you conquered this year:



/tmp/ipykernel_11973/3177650383.py:79: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df['date'] = pd.to_datetime(df['timestamp'], unit='s')


,titleSlug,title,first_solved_date
0,arranging-coins,Arranging Coins,2026-02-18 17:50:55
1,first-bad-version,First Bad Version,2026-02-19 13:53:35
2,find-the-score-difference-in-a-game,Find the Score Difference in a Game,2026-02-22 19:43:07
3,h-index,H-Index,2026-02-23 14:06:33
4,h-index-ii,H-Index II,2026-02-23 14:07:10
...,...,...,...
86,multi-source-flood-fill,Multi Source Flood Fill,2026-04-19 03:17:27
87,employee-importance,Employee Importance,2026-04-19 04:41:13
88,count-sub-islands,Count Sub Islands,2026-04-19 05:13:40
89,network-delay-time,Network Delay Time,2026-04-21 16:47:44


In [3]:
if 'distinct_2026_problems' in locals() and not distinct_2026_problems.empty:
    distinct_2026_problems['month'] = distinct_2026_problems['first_solved_date'].dt.strftime('%B')

    monthly_stats = distinct_2026_problems.groupby(['month']).size().reset_index(name='Count')

    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December']
    monthly_stats['month'] = pd.Categorical(monthly_stats['month'], categories=month_order, ordered=True)
    monthly_stats = monthly_stats.sort_values('month')

    print("--- Distinct Problems Solved by Month in 2026 ---")
    display(monthly_stats)
else:
    print("No 2026 problem data available to group.")

--- Distinct Problems Solved by Month in 2026 ---


,month,Count
1,February,16
2,March,38
0,April,37


In [4]:
if 'distinct_2026_problems' in locals():
    distinct_2026_problems['day_of_week'] = distinct_2026_problems['first_solved_date'].dt.day_name()
    dow_stats = distinct_2026_problems.groupby('day_of_week').size().reindex([
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
    ]).reset_index(name='Count')

    monthly_stats['Growth_Rate'] = monthly_stats['Count'].pct_change().map('{:.2%}'.format).replace('nan%', 'N/A')

    print("--- Key Insights for 2026 ---")
    print(f"\n1. Most Productive Month: {monthly_stats.loc[monthly_stats['Count'].idxmax(), 'month']} ({monthly_stats['Count'].max()} solves)")

    print("\n2. Monthly Velocity & Growth:")
    display(monthly_stats)

    print("\n3. Day of the Week Distribution (Where do you solve most?):")
    display(dow_stats)

    total_days = (distinct_2026_problems['first_solved_date'].max() - distinct_2026_problems['first_solved_date'].min()).days
    if total_days > 0:
        weekly_velocity = len(distinct_2026_problems) / (total_days / 7)
        print(f"\n4. Consistency Check: You are averaging approx {weekly_velocity:.2f} new problems per week.")
else:
    print("Data not found.")

--- Key Insights for 2026 ---

1. Most Productive Month: March (38 solves)

2. Monthly Velocity & Growth:


,month,Count,Growth_Rate
1,February,16,N/A
2,March,38,137.50%
0,April,37,-2.63%



3. Day of the Week Distribution (Where do you solve most?):


,day_of_week,Count
0,Monday,12
1,Tuesday,5
2,Wednesday,5
3,Thursday,9
4,Friday,15
5,Saturday,27
6,Sunday,18



4. Consistency Check: You are averaging approx 9.37 new problems per week.


In [5]:
import os
os.makedirs('templates', exist_ok=True)

html_content = """
<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>LeetCode Analytics 2026</title>
    <link rel='stylesheet' href='https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css'>
    <link href='https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap' rel='stylesheet'>
    <style>
        body { background-color: #f3f4f6; font-family: 'Inter', sans-serif; color: #1f2937; }
        .navbar { background-color: #111827; margin-bottom: 40px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
        .navbar-brand { font-weight: 700; color: #ffffff !important; }
        .stat-card { border: none; border-radius: 12px; transition: transform 0.2s; background: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); }
        .stat-card:hover { transform: translateY(-5px); }
        .stat-label { color: #6b7280; font-size: 0.875rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.025em; }
        .stat-value { font-size: 2.25rem; font-weight: 700; color: #111827; margin: 10px 0; }
        .table-container { background: white; border-radius: 12px; padding: 20px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); }
        .section-title { font-weight: 700; font-size: 1.25rem; margin-bottom: 20px; color: #374151; border-left: 4px solid #3b82f6; padding-left: 15px; }
        .table { margin-bottom: 0; }
        .table thead th { background-color: #f9fafb; border-bottom: 2px solid #e5e7eb; color: #4b5563; font-weight: 600; }
        .border-blue { border-top: 5px solid #3b82f6; }
        .border-green { border-top: 5px solid #10b981; }
        .border-purple { border-top: 5px solid #8b5cf6; }
    </style>
</head>
<body>
    <nav class='navbar navbar-dark'>
        <div class='container'>
            <span class='navbar-brand'>LeetCode 2026 Progress Dashboard</span>
        </div>
    </nav>

    <div class='container'>
        <div class='row g-4'>
            <div class='col-md-4'>
                <div class='card stat-card border-blue p-4'>
                    <div class='stat-label'>New Problems Solved</div>
                    <div class='stat-value'>{{ total_count }}</div>
                    <div class='text-muted small'>Unique ACs in 2026</div>
                </div>
            </div>
            <div class='col-md-4'>
                <div class='card stat-card border-green p-4'>
                    <div class='stat-label'>Weekly Velocity</div>
                    <div class='stat-value'>{{ "%.1f"|format(velocity) }}</div>
                    <div class='text-muted small'>Problems / week</div>
                </div>
            </div>
            <div class='col-md-4'>
                <div class='card stat-card border-purple p-4'>
                    <div class='stat-label'>Peak Month</div>
                    <div class='stat-value'>{{ top_month }}</div>
                    <div class='text-muted small'>Highest activity</div>
                </div>
            </div>
        </div>

        <div class='row g-4 mt-2'>
            <div class='col-lg-7'>
                <div class='table-container'>
                    <h2 class='section-title'>Monthly Performance</h2>
                    {{ monthly_table | safe }}
                </div>
            </div>
            <div class='col-lg-5'>
                <div class='table-container'>
                    <h2 class='section-title'>Activity by Day</h2>
                    {{ dow_table | safe }}
                </div>
            </div>
        </div>
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

In [6]:
from flask import Flask, render_template
from google.colab.output import eval_js
import threading
import traceback

app = Flask(__name__)

@app.route('/')
def index():
    try:
        return render_template('index.html',
                               total_count=len(distinct_2026_problems),
                               velocity=weekly_velocity,
                               top_month=monthly_stats.loc[monthly_stats['Count'].idxmax(), 'month'],
                               monthly_table=monthly_stats.to_html(classes='table table-striped table-hover', index=False),
                               dow_table=dow_stats.to_html(classes='table table-striped table-hover', index=False))
    except Exception as e:
        print("Error in Flask route:")
        traceback.print_exc()
        return str(e), 500

def run_app():
    app.run(port=5000)

try:
    !fuser -k 5000/tcp
except:
    pass

thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

proxy_url = eval_js("google.colab.kernel.proxyPort(5000)")
print(f"Click the link below to view your UI: {proxy_url}")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Click the link below to view your UI: https://5000-m-s-kkb-use4a2-3okm6i613gan4-a.us-east4-2.prod.colab.dev


In [1]:
import os

try:
    !fuser -k 5000/tcp
    print("UI server stopped successfully.")
except Exception as e:
    print(f"Error stopping server: {e}")

UI server stopped successfully.
